# DataAnalyst - 智能数据分析助手

> 基于 HelloAgents 框架的三阶段多智能体数据分析流水线：**规划 → 分析 → 报告**，一键把任意 CSV 变成图文并茂的数据分析报告。

## 项目简介

数据分析是业务决策的重要环节，但人工分析耗时长、容易遗漏数据中的关键模式。DataAnalyst 让你只需**替换一个 CSV 文件**，即可自动完成：数据探查 → 分析任务规划 → 多工具深度分析 → 自动生成图表 → 撰写 Markdown 分析报告。

## 架构设计

```
                 ┌─────────────────────────────────────────────┐
  sales_data.csv │                                             │
  (任意CSV) ───► │  阶段1 规划师Planner(ReActAgent)             │
                 │   └─ 调用 data_overview 探查数据             │
                 │   └─ 输出 3~5 个分析任务(JSON)               │
                 │                                             │
                 │  阶段2 分析员Analyst(ReActAgent)             │
                 │   └─ 逐任务调用6个分析工具(统计/相关性/      │
                 │      异常检测/聚合/绘图)并给出数字结论       │
                 │                                             │
                 │  阶段3 撰写师Reporter(SimpleAgent)           │
                 │   └─ 汇总结论 → Markdown报告 + 嵌入图表      │
                 └─────────────────────────────────────────────┘
                                │
                                ▼
              outputs/analysis_report.md + outputs/charts/*.png
```

## 作者信息
- 姓名：夏明浩
- GitHub：[@minghaoxia61-web](https://github.com/minghaoxia61-web)
- 日期：2026-09-19

## 第1部分：环境配置

In [ ]:
# 导入必要的库
import os
import json
import re
import glob
import warnings
from typing import Any, Dict, List

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from hello_agents import HelloAgentsLLM, SimpleAgent, ReActAgent, ToolRegistry, Config
from hello_agents.tools import Tool, ToolParameter, ToolResponse, ToolErrorCode

warnings.filterwarnings("ignore")

# 加载 .env 中的 LLM 配置（LLM_MODEL_ID / LLM_API_KEY / LLM_BASE_URL，参考 .env.example）
load_dotenv()

# matplotlib 中文显示设置（Windows 用微软雅黑，macOS/Linux 自动回退）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# 项目路径约定
DATA_PATH = "data/sales_data.csv"      # 待分析的数据集：换成你自己的 CSV 即可，无需改任何代码
OUTPUT_DIR = "outputs"                 # 分析报告与图表的输出目录
CHART_DIR = os.path.join(OUTPUT_DIR, "charts")
os.makedirs(CHART_DIR, exist_ok=True)

if not os.getenv("LLM_API_KEY"):
    print("⚠️ 未检测到 LLM_API_KEY，请先复制 .env.example 为 .env 并填入你的 API 密钥")
else:
    print("✅ 环境配置完成，LLM 模型:", os.getenv("LLM_MODEL_ID", "未设置(将使用框架默认值)"))

## 第2部分：数据准备

本项目自带一份模拟电商销售数据（800 条订单，含季节性、地区差异、渠道差异、少量缺失值与异常大额订单）。
**要分析你自己的数据，只需把 CSV 放到 `data/` 目录并修改上面的 `DATA_PATH`。**

In [ ]:
# 读取数据集
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
GLOBAL_DF = df   # 工具层共享的数据引用

print(f"数据集规模: {df.shape[0]} 行 × {df.shape[1]} 列")
df.head()

## 第3部分：数据分析工具定义

基于 hello-agents 的 `Tool` 基类实现 6 个数据分析工具，每个工具负责一类原子分析能力，返回 `ToolResponse`（LLM 阅读的文本 + 结构化数据）：

| 工具 | 功能 |
|---|---|
| `data_overview` | 数据概览：行列数、类型、缺失率、唯一值、数值列统计摘要 |
| `column_profile` | 单列画像：数值列统计量 / 类别列频次 Top 榜 |
| `correlation_analysis` | 数值列两两皮尔逊相关系数，按绝对值排序 |
| `group_aggregate` | 按类别列分组聚合（sum/mean/count...），返回 Top N |
| `detect_outliers` | IQR 异常值检测：阈值、数量、最大异常样本 |
| `plot_chart` | 6 种统计图表（直方图/柱状/箱线/折线/散点/热力图），自动处理中文字体并保存 PNG |

In [ ]:
# ========================================
# 工具1-3：概览 / 列画像 / 相关性
# ========================================
class DataOverviewTool(Tool):
    """输出数据集整体概况，供规划智能体了解数据结构"""

    def __init__(self):
        super().__init__(
            name="data_overview",
            description="获取数据集整体概况：行列数、每列类型/缺失/唯一值、数值列统计摘要。分析开始前应先调用本工具。",
        )

    def get_parameters(self) -> List[ToolParameter]:
        return []

    def run(self, parameters: Dict[str, Any]) -> ToolResponse:
        df = GLOBAL_DF
        lines = [f"数据规模: {df.shape[0]} 行 × {df.shape[1]} 列", "", "字段概况:"]
        for col in df.columns:
            s = df[col]
            missing = int(s.isna().sum())
            miss_pct = missing / len(df) * 100
            uniq = int(s.nunique(dropna=True))
            line = f"- {col} | 类型:{s.dtype} | 缺失:{missing}({miss_pct:.1f}%) | 唯一值:{uniq}"
            if uniq <= 8:
                tops = s.value_counts().head(3)
                line += " | 高频值: " + ", ".join(f"{k}({v})" for k, v in tops.items())
            lines.append(line)
        num_cols = df.select_dtypes(include="number").columns.tolist()
        if num_cols:
            lines.append("")
            lines.append("数值列统计摘要:")
            lines.append(df[num_cols].describe().T.round(2).to_string())
        return ToolResponse.success(text="\n".join(lines))


class ColumnProfileTool(Tool):
    """深入分析单个指定列"""

    def __init__(self):
        super().__init__(
            name="column_profile",
            description="深入分析单个指定列：数值列返回均值/标准差/分位数/偏度，类别列返回频次Top榜单，日期列返回时间范围。",
        )

    def get_parameters(self) -> List[ToolParameter]:
        return [ToolParameter(name="column", type="string",
                              description="要分析的列名（须与数据集列名完全一致）", required=True)]

    def run(self, parameters: Dict[str, Any]) -> ToolResponse:
        col = parameters.get("column", "")
        df = GLOBAL_DF
        if col not in df.columns:
            return ToolResponse.error(code=ToolErrorCode.INVALID_PARAM,
                                      message=f"列不存在: {col}。可用列: {list(df.columns)}")
        s = df[col]
        lines = [f"列 {col} 画像（非空 {int(s.notna().sum())} / {len(s)}）"]
        if pd.api.types.is_datetime64_any_dtype(s):
            lines.append(f"时间范围: {s.min()} ~ {s.max()}")
        elif pd.api.types.is_numeric_dtype(s):
            lines.append(s.describe().round(2).to_string())
            lines.append(f"偏度: {s.skew():.2f}")
        else:
            vc = s.value_counts().head(8)
            lines.append("频次Top8:")
            for k, v in vc.items():
                lines.append(f"- {k}: {v} ({v / len(s) * 100:.1f}%)")
        return ToolResponse.success(text="\n".join(lines))


class CorrelationTool(Tool):
    """数值列两两相关性分析"""

    def __init__(self):
        super().__init__(
            name="correlation_analysis",
            description="计算所有数值列两两之间的皮尔逊相关系数，返回相关性最强的字段对（按绝对值降序）。用于发现字段间的线性关联。",
        )

    def get_parameters(self) -> List[ToolParameter]:
        return [ToolParameter(name="top_n", type="integer", description="返回相关性最强的前N对字段，默认10", required=False)]

    def run(self, parameters: Dict[str, Any]) -> ToolResponse:
        num = GLOBAL_DF.select_dtypes(include="number")
        if num.shape[1] < 2:
            return ToolResponse.error(code=ToolErrorCode.INVALID_PARAM, message="数值列不足2列，无法计算相关性")
        corr = num.corr().round(3)
        pairs = []
        cols = corr.columns
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                pairs.append((cols[i], cols[j], corr.iloc[i, j]))
        pairs.sort(key=lambda x: abs(x[2]), reverse=True)
        top_n = int(parameters.get("top_n") or 10)
        lines = ["相关性最强的字段对（皮尔逊系数）:"]
        for a, b, r in pairs[:top_n]:
            strength = "强" if abs(r) >= 0.7 else ("中等" if abs(r) >= 0.4 else "弱")
            lines.append(f"- {a} × {b}: {r:+.3f}（{strength}{'负' if r < 0 else '正'}相关）")
        return ToolResponse.success(text="\n".join(lines), data={"matrix": corr.to_dict()})

print("✅ 工具1-3定义完成：data_overview / column_profile / correlation_analysis")

In [ ]:
# ========================================
# 工具4-6：分组聚合 / 异常检测 / 绘图
# ========================================
class GroupAggregateTool(Tool):
    """按类别列分组聚合统计"""

    def __init__(self):
        super().__init__(
            name="group_aggregate",
            description="按某个类别列分组，对某个数值列做聚合统计（sum/mean/count/max/min），返回Top N分组结果。适合对比不同类别/地区/渠道的指标。",
        )

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(name="group_col", type="string", description="分组列名（类别列，如：地区、销售渠道、产品类别）", required=True),
            ToolParameter(name="value_col", type="string", description="被聚合的数值列名（如：销售额）", required=True),
            ToolParameter(name="agg", type="string", description="聚合方式: sum/mean/count/max/min，默认sum", required=False),
            ToolParameter(name="top_n", type="integer", description="返回前N组，默认10", required=False),
        ]

    def run(self, parameters: Dict[str, Any]) -> ToolResponse:
        df = GLOBAL_DF
        g, v = parameters.get("group_col", ""), parameters.get("value_col", "")
        agg = (parameters.get("agg") or "sum").lower()
        top_n = int(parameters.get("top_n") or 10)
        if g not in df.columns or v not in df.columns:
            return ToolResponse.error(code=ToolErrorCode.INVALID_PARAM,
                                      message=f"列不存在。可用列: {list(df.columns)}")
        if agg not in {"sum", "mean", "count", "max", "min"}:
            return ToolResponse.error(code=ToolErrorCode.INVALID_PARAM, message=f"不支持的聚合方式: {agg}")
        res = df.groupby(g)[v].agg(agg).sort_values(ascending=False)
        share = (res / res.sum() * 100).round(1) if agg == "sum" else None
        lines = [f"按 {g} 分组对 {v} 做 {agg}（Top {min(top_n, len(res))}）:"]
        for i, (k, val) in enumerate(res.head(top_n).items(), 1):
            s = f" | 占比 {share[k]}%" if share is not None else ""
            lines.append(f"{i}. {k}: {round(float(val), 2)}{s}")
        return ToolResponse.success(text="\n".join(lines), data={"result": res.head(top_n).to_dict()})


class OutlierTool(Tool):
    """IQR 异常值检测"""

    def __init__(self):
        super().__init__(
            name="detect_outliers",
            description="用IQR方法检测指定数值列的异常值：返回正常范围阈值、异常点数量与占比、最大的异常样本。",
        )

    def get_parameters(self) -> List[ToolParameter]:
        return [ToolParameter(name="column", type="string", description="要检测异常值的数值列名", required=True)]

    def run(self, parameters: Dict[str, Any]) -> ToolResponse:
        col = parameters.get("column", "")
        df = GLOBAL_DF
        if col not in df.columns:
            return ToolResponse.error(code=ToolErrorCode.INVALID_PARAM,
                                      message=f"列不存在。可用列: {list(df.columns)}")
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mask = (s < low) | (s > high)
        out = df.loc[s[mask].index]
        lines = [f"列 {col} 的IQR异常检测:",
                 f"- 正常范围: [{low:.2f}, {high:.2f}]（Q1={q1:.2f}, Q3={q3:.2f}）",
                 f"- 异常点: {len(out)} 个，占比 {len(out) / len(df) * 100:.2f}%"]
        if len(out):
            id_cols = [c for c in df.columns if ("ID" in c or "日期" in c) and c != col][:2]
            top = out.sort_values(col, ascending=False).head(5)
            lines.append("- 最大的异常样本:")
            lines.append(top[id_cols + [col]].to_string(index=False))
        return ToolResponse.success(text="\n".join(lines))


_CHART_SEQ = {"n": 0}   # 图表编号（避免文件名冲突）

def _safe_name(x: str) -> str:
    """把列名转成安全的文件名片段"""
    return "".join(c if c.isalnum() else "_" for c in str(x))[:20] or "chart"


class PlotChartTool(Tool):
    """生成统计图表并保存 PNG，返回可嵌入 Markdown 的相对路径"""

    def __init__(self):
        super().__init__(
            name="plot_chart",
            description="生成统计图表并保存为PNG（中文可正常显示）。类型: histogram/bar/box/line/scatter/heatmap。返回图片相对路径（相对outputs目录），可直接嵌入Markdown报告。",
        )

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(name="chart_type", type="string",
                          description="图表类型: histogram/bar/box/line/scatter/heatmap", required=True),
            ToolParameter(name="x_column", type="string", description="X轴列名（heatmap可留空）", required=False),
            ToolParameter(name="y_column", type="string", description="Y轴数值列名（histogram/box/heatmap可留空）", required=False),
            ToolParameter(name="agg", type="string", description="bar/line图的聚合方式: sum/mean/count，默认sum", required=False),
            ToolParameter(name="title", type="string", description="图表标题（中文），默认自动生成", required=False),
        ]

    def run(self, parameters: Dict[str, Any]) -> ToolResponse:
        df = GLOBAL_DF
        ctype = (parameters.get("chart_type") or "").lower()
        x, y = parameters.get("x_column") or "", parameters.get("y_column") or ""
        agg = (parameters.get("agg") or "sum").lower()
        title = parameters.get("title") or ""
        num_cols = df.select_dtypes(include="number").columns.tolist()

        if ctype not in {"histogram", "bar", "box", "line", "scatter", "heatmap"}:
            return ToolResponse.error(code=ToolErrorCode.INVALID_PARAM, message=f"不支持的图表类型: {ctype}")
        if ctype != "heatmap":   # 校验列名
            need = {"histogram": [], "box": [x or y], "bar": [x, y], "line": [x, y], "scatter": [x, y]}[ctype]
            for c in [c for c in need if c]:
                if c not in df.columns:
                    return ToolResponse.error(code=ToolErrorCode.INVALID_PARAM,
                                              message=f"列不存在: {c}。可用列: {list(df.columns)}")
        if agg not in {"sum", "mean", "count"}:
            agg = "sum"

        fig, ax = plt.subplots(figsize=(8, 5))
        if ctype == "heatmap":
            corr = df[num_cols].corr()
            im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
            ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
            ax.set_yticks(range(len(corr.columns)), corr.columns)
            for i in range(len(corr.columns)):
                for j in range(len(corr.columns)):
                    ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
            fig.colorbar(im, ax=ax, shrink=0.8)
            title = title or "数值列相关性热力图"
        elif ctype == "histogram":
            if not num_cols:
                return ToolResponse.error(code=ToolErrorCode.EXECUTION_ERROR, message="数据集中没有数值列，无法绘制直方图")
            col = y or x or num_cols[0]
            data = pd.to_numeric(df[col], errors="coerce").dropna()
            if data.empty:
                return ToolResponse.error(code=ToolErrorCode.EXECUTION_ERROR,
                                          message=f"列 {col} 无法转换为数值，无法绘制直方图")
            ax.hist(data, bins=30, color="#4C72B0", edgecolor="white")
            ax.set_xlabel(col)
            ax.set_ylabel("频数")
            title = title or f"{col} 的分布直方图"
        elif ctype == "box":
            col = y or x
            data = pd.to_numeric(df[col], errors="coerce").dropna()
            if data.empty:
                return ToolResponse.error(code=ToolErrorCode.EXECUTION_ERROR,
                                          message=f"列 {col} 无法绘制箱线图（非数值列或无有效数据）")
            ax.boxplot(data, vert=True, tick_labels=[col])
            ax.set_ylabel(col)
            title = title or f"{col} 的箱线图"
        elif ctype == "bar":
            tmp = df.dropna(subset=[x])
            if agg == "count" or not y:
                res = tmp.groupby(x).size().sort_values(ascending=False)
                ylab = "数量"
            else:
                res = tmp.groupby(x)[y].agg(agg).sort_values(ascending=False)
                ylab = f"{y}（{agg}）"
            res = res.head(10)
            ax.barh([str(k) for k in res.index][::-1], list(res.values)[::-1], color="#4C72B0")
            ax.set_xlabel(ylab)
            title = title or f"各{x}的{ylab}对比（Top{len(res)}）"
        elif ctype == "line":
            tmp = df.dropna(subset=[x, y]).copy()
            xt = pd.to_datetime(tmp[x], errors="coerce")
            if xt.notna().sum() > len(tmp) * 0.8:      # 日期列：按月重采样
                tmp["_t"] = xt
                res = tmp.set_index("_t")[y].resample("ME").agg(agg)
                ax.plot(res.index, res.values, marker="o", color="#4C72B0")
                ax.set_xlabel(x + "（按月）")
            else:
                res = tmp.groupby(x)[y].agg(agg).sort_index()
                ax.plot([str(k) for k in res.index], res.values, marker="o", color="#4C72B0")
                ax.set_xlabel(x)
            ax.set_ylabel(f"{y}（{agg}）")
            title = title or f"{y}随{x}的变化趋势（{agg}）"
        else:                                          # scatter
            tmp = df.dropna(subset=[x, y])
            if tmp.empty:
                return ToolResponse.error(code=ToolErrorCode.EXECUTION_ERROR,
                                          message=f"列 {x} 或 {y} 无有效成对数据，无法绘制散点图")
            if len(tmp) > 2000:
                tmp = tmp.sample(2000, random_state=42)
            ax.scatter(tmp[x], tmp[y], s=12, alpha=0.5, color="#4C72B0")
            ax.set_xlabel(x)
            ax.set_ylabel(y)
            title = title or f"{x} 与 {y} 的散点图"
        ax.set_title(title)
        ax.grid(alpha=0.25)

        _CHART_SEQ["n"] += 1
        # 文件名使用实际绘制的列（箱线图/直方图取被统计列，热力图为corr）
        name_col = {"heatmap": "corr", "histogram": (y or x or (num_cols[0] if num_cols else "data")),
                    "box": (y or x), "bar": x, "line": x, "scatter": x}[ctype]
        fname = f"chart_{_CHART_SEQ['n']}_{ctype}_{_safe_name(name_col)}.png"
        rel = f"charts/{fname}"
        fig.savefig(os.path.join(CHART_DIR, fname), dpi=150, bbox_inches="tight")
        plt.close("all")
        return ToolResponse.success(
            text=(f"图表已生成: {rel}（相对 {OUTPUT_DIR}/ 目录）\n标题: {title}\n"
                  f"请在报告对应小节用 ![图表描述]({rel}) 嵌入该图片。"),
            data={"path": rel},
        )

print("✅ 工具4-6定义完成：group_aggregate / detect_outliers / plot_chart")

## 第4部分：智能体构建

| 智能体 | 范式 | 职责 | 配备工具 |
|---|---|---|---|
| 分析规划师 Planner | ReActAgent | 探查数据、规划 3~5 个分析任务（JSON） | data_overview、column_profile |
| 数据分析员 Analyst | ReActAgent | 逐任务调用工具完成分析并给出数字结论 | 全部 6 个工具 |
| 报告撰写师 Reporter | SimpleAgent | 汇总结论，撰写图文并茂的 Markdown 报告 | 无（纯生成） |

In [ ]:
# ========================================
# 工具注册表：规划智能体配轻量探查工具，分析智能体配全部分析工具
# ========================================
planner_registry = ToolRegistry()
planner_registry.register_tool(DataOverviewTool())
planner_registry.register_tool(ColumnProfileTool())

analysis_registry = ToolRegistry()
for t in [DataOverviewTool(), ColumnProfileTool(), CorrelationTool(),
          GroupAggregateTool(), OutlierTool(), PlotChartTool()]:
    analysis_registry.register_tool(t)

# LLM 客户端（自动读取 .env 中的 LLM_MODEL_ID / LLM_API_KEY / LLM_BASE_URL）
llm = HelloAgentsLLM()

# 说明: hello-agents 1.0.0 的 TraceLogger 在每次 run() 结束时关闭轨迹文件，
# 同一智能体第二次 run() 时会因写入已关闭的文件句柄而报错（I/O operation on closed file）。
# 流水线需要对同一智能体多轮调用，因此这里关闭轨迹追踪（不影响分析与报告结果）。
agent_config = Config(trace_enabled=False)

PLANNER_PROMPT = """你是一位资深数据分析规划师，负责为已加载到内存中的CSV数据集制定分析计划。

你的工作流程：
1. 先调用 data_overview 工具了解数据集的字段结构与数据质量（如有需要可再用 column_profile 查看关键列）
2. 结合字段实际含义，规划 3~5 个有业务价值的分析任务，可覆盖：时间趋势、类别结构、分组对比、相关性分析、异常值检测等维度
3. 每个任务都要具体可执行：写明使用哪个字段、做什么统计、需要什么图表（图表类型支持: histogram/bar/box/line/scatter/heatmap）

最后，你必须以纯JSON数组作为最终答案输出分析计划（直接输出JSON本身，不要加代码块围栏，不要输出任何解释文字），格式如下:
[
  {"task": "任务简短标题", "goal": "具体分析目标，写明字段、统计方式与所需图表"}
]"""

ANALYST_PROMPT = """你是一位严谨的数据分析员。针对交给你的每个分析任务：

1. 选择合适的工具完成分析（group_aggregate / correlation_analysis / detect_outliers / column_profile / plot_chart）
2. 需要展示分布、对比或趋势时，请调用 plot_chart 生成图表（标题用中文）
3. 最终用 3~6 句话总结结论，结论中必须引用工具返回的具体数字，不要空泛
4. 如果生成了图表，在结论最后单独一行列出图片路径，格式: 图表: charts/xxx.png

注意：一个任务通常 1~3 次工具调用即可完成，不要重复调用完全相同的工具。"""

REPORTER_PROMPT = """你是一位资深数据分析师，负责把分析结论整理成一份专业的中文数据分析报告（Markdown格式）。

报告结构要求：
# 报告标题
## 一、数据概况
## 二、核心发现（每个分析任务一个小节，标题概括发现，正文给出结论与关键数字）
## 三、业务建议（基于发现给出 3~5 条可落地的建议）
## 四、分析方法说明（简述使用的工具与统计方法）

撰写要求：
1. 所有结论与数字必须来自输入内容，禁止编造数据
2. 若某条结论中带有"图表: charts/xxx.png"路径，请在对应小节用Markdown图片语法嵌入: ![图表描述](charts/xxx.png)
3. 语言专业、简洁，突出业务洞察"""

planner_agent = ReActAgent(name="分析规划师", llm=llm, tool_registry=planner_registry,
                           system_prompt=PLANNER_PROMPT, config=agent_config, max_steps=5)
analyst_agent = ReActAgent(name="数据分析员", llm=llm, tool_registry=analysis_registry,
                           system_prompt=ANALYST_PROMPT, config=agent_config, max_steps=6)
reporter_agent = SimpleAgent(name="报告撰写师", llm=llm, system_prompt=REPORTER_PROMPT,
                             config=agent_config)

print("✅ 三个智能体构建完成：分析规划师(ReAct) / 数据分析员(ReAct) / 报告撰写师(Simple)")

## 第5部分：三阶段分析流水线

`run_pipeline()` 串起三个智能体：**规划 → 逐任务分析 → 汇总报告**，并把报告与图表落盘到 `outputs/` 目录。

In [ ]:
# ========================================
# 流水线实现
# ========================================
def parse_tasks(plan_text: str) -> List[Dict[str, str]]:
    """从规划智能体的输出中解析任务列表（JSON优先，正则兜底）"""
    candidates = []
    m = re.search(r"```(?:json)?\s*(\[.*?\])\s*```", plan_text, re.S)
    if m:
        candidates.append(m.group(1))
    # 兜底1: 输出是裸JSON数组（没有代码块围栏）
    if not candidates:
        i, j = plan_text.find("["), plan_text.rfind("]")
        if 0 <= i < j:
            candidates.append(plan_text[i:j + 1])
    for cand in candidates:
        try:
            tasks = json.loads(cand)
            return [
                {"task": str(t.get("task", "")).strip() or f"任务{i}",
                 "goal": str(t.get("goal", "")).strip()}
                for i, t in enumerate(tasks, 1) if isinstance(t, dict)
            ]
        except json.JSONDecodeError:
            continue
    # 兜底2: 按"任务N：描述"格式的行提取
    tasks = []
    for line in plan_text.splitlines():
        m2 = re.match(r"\s*[-*\d.、)]*\s*(?:\*\*)?任务\d+[**:：]?\s*(.+)", line)
        if m2 and len(m2.group(1).strip()) > 4:
            tasks.append({"task": m2.group(1).strip(), "goal": ""})
    return tasks[:6]


def run_pipeline(data_path: str = DATA_PATH,
                 report_path: str = os.path.join(OUTPUT_DIR, "analysis_report.md")):
    """三阶段分析流水线：规划 → 逐任务分析 → 汇总报告"""
    print("=" * 60)
    print("【阶段1/3】分析规划：探查数据并生成分析任务")
    plan_text = planner_agent.run(
        f"请针对数据集 {data_path}（{GLOBAL_DF.shape[0]} 行 × {GLOBAL_DF.shape[1]} 列，字段: "
        f"{', '.join(GLOBAL_DF.columns)}）制定分析计划。"
    )
    tasks = parse_tasks(plan_text)
    if not tasks:
        tasks = [{"task": "数据整体概览与质量检查", "goal": "使用 data_overview 了解数据"}]
    print(f"\n✅ 规划完成，共 {len(tasks)} 个分析任务:")
    for i, t in enumerate(tasks, 1):
        print(f"   任务{i}: {t['task']}")

    print("\n" + "=" * 60)
    print("【阶段2/3】逐任务深度分析")
    conclusions = []
    for i, t in enumerate(tasks, 1):
        print(f"\n>>> 执行任务{i}: {t['task']}")
        out = analyst_agent.run(f"分析任务: {t['task']}\n分析目标: {t['goal'] or t['task']}")
        conclusions.append({"task": t["task"], "conclusion": out})
        print(f"✅ 任务{i} 完成")

    print("\n" + "=" * 60)
    print("【阶段3/3】撰写分析报告")
    chart_files = sorted(glob.glob(os.path.join(CHART_DIR, "*.png")))
    chart_paths = [os.path.relpath(p, OUTPUT_DIR).replace("\\\\", "/") for p in chart_files]
    report_input = (
        f"数据集概况: {GLOBAL_DF.shape[0]} 行 × {GLOBAL_DF.shape[1]} 列，字段: {', '.join(GLOBAL_DF.columns)}\n\n"
        f"已生成的图表文件（相对outputs目录）: {json.dumps(chart_paths, ensure_ascii=False)}\n\n"
        f"各分析任务的结论（JSON）:\n{json.dumps(conclusions, ensure_ascii=False, indent=2)}\n\n"
        f"请撰写完整的数据分析报告。"
    )
    report = reporter_agent.run(report_input)
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report)
    print(f"\n✅ 报告已保存: {report_path}")
    print(f"✅ 共生成图表 {len(chart_paths)} 张，保存于 {CHART_DIR}/")
    return report, tasks, conclusions


print("✅ 流水线函数定义完成")

## 第6部分：运行完整分析

执行三阶段流水线（需要 `.env` 中配置好 LLM API 密钥）。运行结束后：
- 报告：`outputs/analysis_report.md`
- 图表：`outputs/charts/*.png`

In [ ]:
report, tasks, conclusions = run_pipeline()

In [ ]:
# 查看生成的分析报告
print(report)

## 第7部分：工具自检（不消耗 LLM 调用）

直接调用两个核心工具验证工具层工作正常——即使没有配置 API 密钥，也可以运行本单元格体验工具层。

In [ ]:
# 自检1：分组聚合 —— 各产品类别的销售额贡献
print(GroupAggregateTool().run({"group_col": "产品类别", "value_col": "销售额"}).text)

print("\n" + "=" * 50 + "\n")

# 自检2：IQR异常值检测 —— 找出异常大额订单
print(OutlierTool().run({"column": "销售额"}).text)

## 第8部分：总结与展望

### 实现的功能
- ✅ 通用 CSV 数据分析：替换数据文件即可分析新数据集，无需改代码
- ✅ 三阶段多智能体流水线：规划（Plan）→ 分析（Execute）→ 报告（Report）
- ✅ 6 个原子分析工具：概览 / 列画像 / 相关性 / 分组聚合 / 异常检测 / 绘图
- ✅ 自动中文图表生成，并在报告中以相对路径嵌入
- ✅ 工具层可独立运行（第7部分自检），便于调试与评审

### 遇到的挑战与解决方案
- **规划输出不稳定**：LLM 偶尔不按 JSON 输出 → 采用「JSON 优先 + 正则兜底」双解析策略（`parse_tasks`）
- **matplotlib 中文乱码**：统一配置 `font.sans-serif` 候选字体链，并处理负号显示
- **LLM 传参错误**：所有工具对列名/参数做校验并返回可用列提示，智能体可自行纠错重试

### 未来改进方向
- [ ] 支持多 Sheet / Excel 与数据库数据源
- [ ] 引入 ReflectionAgent 对报告质量自动复审
- [ ] 增加 Gradio 交互界面，支持拖拽上传
- [ ] 分析结果缓存，避免重复计算